<a href="https://colab.research.google.com/github/niharnandala/ntsb-safety-review-agent/blob/main/notebooks/01b_data_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01b · Data audit

Every case we opened by hand showed something new: no reply yet, a missing first letter, NTSB using info
that's not in the reply, one letter covering many cases. We can't read 10,000 cases one by one, so instead:
1. **count** every pattern across all the data
2. **read a couple of real examples** of each pattern
3. **decide** what we do with each one (keep / drop / fix / note as a limit)

The decisions go into `02_build_episodes`, which we re-run afterwards.

**Runtime:** CPU.

## Load the data

In [1]:
from google.colab import files
files.upload()   # pick the CAROL zip

!unzip -o -q "carol_recommendations_data (1).zip" -d data

Saving carol_recommendations_data (1).zip to carol_recommendations_data (1).zip


In [2]:
import json, glob, re, random
from collections import Counter
from datetime import date

with open(glob.glob("data/*.json")[0]) as f:
    recs = json.load(f)
print(len(recs), "recommendations")

10000 recommendations


## Helpers
Same basics as before. Plus `show_case`, which prints a whole thread in date order so we can read it.
`examples` collects a few case IDs for every pattern so we can look at them at the end.

In [3]:
GRADE = re.compile(r'["“]\s*((?:Open|Closed)\s*[-–—]+\s*[A-Za-z /]+?)\s*[."”,]', re.I)
ANY_ID = re.compile(r"\b[AHIMPR]\s*-?\s*\d{2}\s*-\s*\d{1,3}\b")

def text(l):
    return (l.get("ResponseSummary") or "").strip()

def from_ntsb(l):
    return bool(l.get("IsFromNtsb")) and not text(l).startswith("-From")

def dated(a):
    # non-empty letters with a date, oldest first
    ls = [l for l in (a.get("Correspondence") or []) if text(l) and l.get("CorrespondenceDate")]
    return sorted(ls, key=lambda l: l["CorrespondenceDate"])

def year_of(l):
    return int(l["CorrespondenceDate"][:4])

def rec_year(srid):
    yy = int(srid.split("-")[1])
    return 1900 + yy if yy > 30 else 2000 + yy

threads = [(r, a) for r in recs for a in r["Addressees"]]
print(len(threads), "threads (recommendation × recipient)")

examples = {}
def add_example(pattern, r, a):
    examples.setdefault(pattern, [])
    if len(examples[pattern]) < 3:
        examples[pattern].append((r["Srid"], a.get("AddressAcronym", "")))

def show_case(srid, who=None, chars=500):
    r = next(r for r in recs if r["Srid"] == srid)
    print("=" * 90)
    print(srid, "|", r.get("Mode"), "|", r.get("Priority"), "| final status:", r.get("OpenClosed"))
    print("ASKED:", (r.get("Subject") or "")[:400])
    for a in r["Addressees"]:
        if who and a.get("AddressAcronym") != who:
            continue
        print("\n--- to", a.get("AddressAcronym"), "| status:", a.get("OpenClosedAddressee"))
        for l in dated(a):
            who_sent = "NTSB" if from_ntsb(l) else "ORG "
            print(f"\n[{l['CorrespondenceDate'][:10]}] {who_sent} · {l.get('CorrespondenceType')}")
            print(text(l)[:chars].replace("\r", "").replace("\n", " "))

18904 threads (recommendation × recipient)


## 1 · What kinds of letters are there?
Each letter has a type (transmittal, official correspondence, reclassification…). Different types might need
different handling, so first we just see what exists and who sends each type.

In [4]:
types = Counter()
for r, a in threads:
    for l in (a.get("Correspondence") or []):
        types[(l.get("CorrespondenceType") or "?", "NTSB" if from_ntsb(l) else "ORG")] += 1

for (t, who), n in types.most_common():
    print(f"{n:7}  {who:4}  {t}")

  54592  NTSB  Official Correspondence
  36594  ORG   Official Correspondence
   5458  NTSB  NPRM Response
   2405  NTSB  Transmittal Letter
   1461  NTSB  Recommendation Reiteration
   1207  NTSB  Recommendation Mention
    373  ORG   ?
    207  NTSB  Report Reclassification
    194  ORG   Non-Recipient Correspondence
    110  NTSB  Non-Recipient Correspondence
    100  NTSB  Staff-Level Communication
     39  ORG   Recommendation Reiteration
     27  ORG   Staff-Level Communication
      9  ORG   NPRM Response
      5  ORG   Report Reclassification


## 2 · What does a thread look like?
A thread = one recommendation sent to one recipient. Possible shapes:
- **empty**: no usable letters at all
- **no reply yet**: only NTSB letters (like A-26-065)
- **reply, but NTSB never answered** after any reply
- **normal**: at least one reply with an NTSB letter after it

For "no reply yet" we also check how many are past the 90-day mark today, since those are the overdue/reminder cases.

In [5]:
shape = Counter()
overdue_now = 0
today = date.today()

for r, a in threads:
    ls = dated(a)
    replies = [i for i, l in enumerate(ls) if not from_ntsb(l)]
    if not ls:
        shape["empty"] += 1
    elif not replies:
        shape["no reply yet"] += 1
        add_example("no reply yet", r, a)
        last = date.fromisoformat(ls[-1]["CorrespondenceDate"][:10])
        if r.get("OpenClosed", "").startswith("Open") and (today - last).days > 90:
            overdue_now += 1
            add_example("open, no reply, overdue now", r, a)
    elif not any(from_ntsb(l) for l in ls[replies[0] + 1:]):
        shape["reply but no NTSB answer"] += 1
        add_example("reply but no NTSB answer", r, a)
    else:
        shape["normal"] += 1

for k, n in shape.most_common():
    print(f"{n:7}  {k}")
print(f"\n{overdue_now} open threads with no reply and >90 days since the last letter")

  13731  normal
   3998  no reply yet
    991  reply but no NTSB answer
    184  empty

755 open threads with no reply and >90 days since the last letter


## 3 · Is the first letter really the start?
Day 0 is supposed to be when NTSB sent the recommendation. But in M-89-073 the first letter in the data was a
reminder from 1992, while the recommendation was from 1989. So we check:
- what type the first letter usually is
- how often the first letter comes 2+ years after the year in the recommendation's ID
- how often the first letter is from the organisation instead of NTSB

In [6]:
first_type = Counter()
late_start = 0
org_first = 0

for r, a in threads:
    ls = dated(a)
    if not ls:
        continue
    first = ls[0]
    first_type[first.get("CorrespondenceType") or "?"] += 1
    if year_of(first) - rec_year(r["Srid"]) >= 2:
        late_start += 1
        add_example("first letter 2+ years after the ID year", r, a)
    if not from_ntsb(first):
        org_first += 1
        add_example("first letter is from the org", r, a)

print("first letter type:")
for t, n in first_type.most_common(8):
    print(f"{n:7}  {t}")
print(f"\n{late_start} threads where the first letter is 2+ years after the ID year")
print(f"{org_first} threads where the first letter is from the organisation")

first letter type:
  15959  Official Correspondence
   2143  Transmittal Letter
    413  NPRM Response
    187  Recommendation Reiteration
      8  Recommendation Mention
      5  Report Reclassification
      5  Non-Recipient Correspondence

4887 threads where the first letter is 2+ years after the ID year
10186 threads where the first letter is from the organisation


## 4 · Grades that change without a new reply
Sometimes NTSB changes a grade without any new reply: e.g. reclassified in a new accident report, or replaced
by a newer recommendation. Those aren't "reply → grade" decisions, so we need to know how often they happen
and what types they are.

In [7]:
no_reply_grade = Counter()

for r, a in threads:
    ls = dated(a)
    for i, l in enumerate(ls):
        if not from_ntsb(l) or not GRADE.search(text(l)):
            continue
        prev = ls[i - 1] if i > 0 else None
        if prev is None or from_ntsb(prev):
            no_reply_grade[l.get("CorrespondenceType") or "?"] += 1
            add_example("grade with no new reply before it", r, a)

print(sum(no_reply_grade.values()), "NTSB letters with a grade but no new reply right before them")
for t, n in no_reply_grade.most_common():
    print(f"{n:7}  {t}")

9560 NTSB letters with a grade but no new reply right before them
   7707  Official Correspondence
   1151  NPRM Response
    486  Recommendation Reiteration
     69  Transmittal Letter
     69  Report Reclassification
     42  Recommendation Mention
     36  Non-Recipient Correspondence


## 5 · Batch letters
How common are letters that talk about lots of recommendations at once (3+ IDs)?

In [8]:
batch_ntsb = batch_org = total_ntsb = total_org = 0
for r, a in threads:
    for l in dated(a):
        many = len(set(ANY_ID.findall(text(l)))) >= 3
        if from_ntsb(l):
            total_ntsb += 1; batch_ntsb += many
        else:
            total_org += 1; batch_org += many
        if many:
            add_example("batch letter (3+ IDs)", r, a)

print(f"NTSB letters naming 3+ recommendations: {batch_ntsb} of {total_ntsb}")
print(f"org letters naming 3+ recommendations:  {batch_org} of {total_org}")

NTSB letters naming 3+ recommendations: 7321 of 59385
org letters naming 3+ recommendations:  1609 of 35547


## 6 · NTSB uses info that isn't in the reply
In M-89-073, NTSB closed the case because of a "Cartographic Order" the Coast Guard never mentioned.
If the model trains on letters like that, it learns to make up facts. This is a rough check: in NTSB's answer,
look for signs of outside info (meetings, phone calls, attachments) and for document names
(Order 002/92, AD 2002-26-13, AC 120-27F…) that don't appear anywhere in the reply.

In [9]:
OUTSIDE = re.compile(r"\b(meeting|telephone|phone call|conversation|discussion with|briefing|enclos\w*|attach\w*)\b", re.I)
DOC = re.compile(r"\b(?:Order|AD|AC|Notice|Bulletin|Circular|SAIB|Directive|Rule)\s+(?:No\.\s*)?[A-Z0-9][\w\-/\.]*\d", re.I)

answers = with_outside = with_new_doc = 0
for r, a in threads:
    ls = dated(a)
    for i, l in enumerate(ls):
        if from_ntsb(l) or i + 1 >= len(ls) or not from_ntsb(ls[i + 1]):
            continue
        reply, ans = text(l), text(ls[i + 1])
        if not GRADE.search(ans):
            continue
        answers += 1
        if OUTSIDE.search(ans):
            with_outside += 1
        docs = {d.lower() for d in DOC.findall(ans)}
        if any(d not in reply.lower() for d in docs):
            with_new_doc += 1
            add_example("NTSB cites a document the reply doesn't", r, a)

print(answers, "graded NTSB answers")
print(with_outside, "mention meetings / calls / attachments")
print(with_new_doc, "cite a document that the reply never mentions")

12738 graded NTSB answers
1029 mention meetings / calls / attachments
577 cite a document that the reply never mentions


## 7 · Recommendations sent to several recipients
Some recommendations go to many organisations (e.g. all 50 states). This matters for the peer comparison feature.

In [10]:
n_recip = Counter(len(r["Addressees"]) for r in recs)
print("recipients per recommendation:")
for k in sorted(n_recip):
    if k <= 5:
        print(f"{n_recip[k]:7}  {k}")
print(f"{sum(v for k, v in n_recip.items() if k > 5):7}  more than 5")
print("max:", max(n_recip))

recipients per recommendation:
   9399  1
    176  2
     82  3
     44  4
     36  5
    263  more than 5
max: 78


## 8 · Very long threads and letters
Tiny Aya's desk is ~8K tokens (~32K characters). How many threads and letters are huge?

In [11]:
long_threads = sum(1 for r, a in threads if len(dated(a)) > 20)
long_letters = sum(1 for r, a in threads for l in dated(a) if len(text(l)) > 8000)
huge_letters = sum(1 for r, a in threads for l in dated(a) if len(text(l)) > 20000)
for r, a in threads:
    if len(dated(a)) > 20:
        add_example("thread with 20+ letters", r, a)

print(long_threads, "threads with more than 20 letters")
print(long_letters, "letters longer than 8,000 characters (~2,000 tokens)")
print(huge_letters, "letters longer than 20,000 characters (~5,000 tokens)")

288 threads with more than 20 letters
3512 letters longer than 8,000 characters (~2,000 tokens)
260 letters longer than 20,000 characters (~5,000 tokens)


## 9 · Data hygiene
Things we already handle, just confirming the size: undated letters, the same letter stored twice in a thread,
and letters flagged as NTSB that are really from the organisation.

In [12]:
undated = dupes = mislabelled = 0
for r, a in threads:
    ls = [l for l in (a.get("Correspondence") or []) if text(l)]
    undated += sum(1 for l in ls if not l.get("CorrespondenceDate"))
    seen = Counter(text(l) for l in ls)
    dupes += sum(n - 1 for n in seen.values() if n > 1)
    mislabelled += sum(1 for l in ls if l.get("IsFromNtsb") and text(l).startswith("-From"))
    if any(n > 1 for n in seen.values()):
        add_example("same letter stored twice", r, a)

print(undated, "letters with no date")
print(dupes, "duplicate copies of a letter inside the same thread")
print(mislabelled, "letters flagged NTSB that start with '-From'")

1869 letters with no date
6511 duplicate copies of a letter inside the same thread
107 letters flagged NTSB that start with '-From'


## 10 · Old vs new writing style
Older letters are often ALL CAPS and short summaries. If most training data is old-style, the model might
struggle with modern letters. Share of ALL CAPS letters per decade:

In [13]:
caps = Counter()
total = Counter()
for r, a in threads:
    for l in dated(a):
        t = text(l)
        letters_only = [c for c in t if c.isalpha()]
        if len(letters_only) < 50:
            continue
        decade = year_of(l) // 10 * 10
        total[decade] += 1
        if sum(c.isupper() for c in letters_only) / len(letters_only) > 0.9:
            caps[decade] += 1

for d in sorted(total):
    print(f"{d}s: {total[d]:6} letters, {100 * caps[d] // total[d]:3}% ALL CAPS")

1960s:      1 letters,   0% ALL CAPS
1970s:      5 letters, 100% ALL CAPS
1980s:   7308 letters,  56% ALL CAPS
1990s:  21329 letters,  80% ALL CAPS
2000s:  27151 letters,   8% ALL CAPS
2010s:  23925 letters,   0% ALL CAPS
2020s:  13573 letters,   0% ALL CAPS
2500s:      1 letters,   0% ALL CAPS
3010s:      1 letters,   0% ALL CAPS


## Examples to read
A few real cases for every pattern. Run the next cell to see the list, then use `show_case("<ID>")` on the ones
we want to read together.

In [14]:
for pattern, ids in examples.items():
    print(f"{pattern}:")
    for srid, who in ids:
        print("   ", srid, "→", who)

no reply yet:
    H-25-025 → TX
    A-26-064 → FAA
    A-26-065 → FAA
open, no reply, overdue now:
    H-25-025 → TX
    A-26-064 → FAA
    A-26-065 → FAA
reply but no NTSB answer:
    A-25-038 → FAA
    A-25-039 → FAA
    A-26-057 → 
first letter is from the org:
    H-25-027 → NAPT
    H-25-021 → ODOT
    A-20-024 → 
first letter 2+ years after the ID year:
    P-19-016 → OH
    P-19-016 → WI
    P-19-016 → NM
grade with no new reply before it:
    H-25-025 → TX
    H-25-027 → NAPT
    H-25-027 → NASDPTS
batch letter (3+ IDs):
    H-25-026 → LISD
    H-25-025 → TX
    H-25-027 → NAPT
NTSB cites a document the reply doesn't:
    H-20-002 → NHTSA
    H-17-038 → NHTSA
    A-16-036 → FAA
thread with 20+ letters:
    H-25-003 → 
    H-25-004 → 
    H-23-020 → 
same letter stored twice:
    H-25-003 → 
    H-25-003 → ODOT
    H-25-004 → 


In [15]:
# read one case in full (change the ID); chars = how much of each letter to show
show_case(examples["grade with no new reply before it"][0][0], chars=600)

H-25-025 | Highway | CLASS I | final status: Open - Await Response
ASKED: TO THE STATE OF TEXAS: Establish, distribute, and require implementation of enforceable policies and procedures for seat belt use on school buses, with routine audits to ensure compliance with Texas Education Code 34.013, for all school districts in Texas so that every student and driver on school buses equipped with passenger seat belts is properly restrained by lap/shoulder belts or five-point h

--- to TX | status: Open - Await Response

[2025-10-28] NTSB · Transmittal Letter
The NTSB is an independent federal agency charged by Congress with investigating every civil aviation accident in the United States and significant events in the other modes of transportation—railroad, transit, highway, marine, pipeline, and commercial space. We determine the probable causes of the accidents and events we investigate and issue safety recommendations aimed at preventing future occurrences. In addition, we conduct transport

## Why does NTSB grade without a new reply?
9,560 graded NTSB letters had no org reply right before them. That's a lot, so we pulled 10 random ones to see why.

In [16]:
random.seed(5)
cases = []
for r, a in threads:
    ls = dated(a)
    for i, l in enumerate(ls):
        if from_ntsb(l) and GRADE.search(text(l)) and i > 0 and from_ntsb(ls[i - 1]):
            cases.append((r["Srid"], a.get("AddressAcronym"), ls, i))

dup = sum(text(ls[i - 1]) == text(ls[i]) for _, _, ls, i in cases)
had_reply_earlier = sum(any(not from_ntsb(x) for x in ls[:i]) for _, _, ls, i in cases)
print(len(cases), "graded NTSB letters right after another NTSB letter")
print(dup, "of them are exact duplicates of the letter before")
print(had_reply_earlier, "have an org reply somewhere earlier in the thread")

for srid, who, ls, i in random.sample(cases, 10):
    prev, l = ls[i - 1], ls[i]
    m = GRADE.search(text(l))
    print("=" * 80)
    print(srid, "→", who)
    print(f"PREVIOUS [{prev['CorrespondenceDate'][:10]}] NTSB · {prev.get('CorrespondenceType')} | duplicate: {text(prev) == text(l)}")
    print("   ", text(prev)[:200].replace("\n", " "))
    print(f"THIS     [{l['CorrespondenceDate'][:10]}] NTSB · {l.get('CorrespondenceType')}")
    print("   …", text(l)[max(0, m.start() - 250): m.end()].replace("\n", " "))

7093 graded NTSB letters right after another NTSB letter
2036 of them are exact duplicates of the letter before
5695 have an org reply somewhere earlier in the thread
H-01-008 → NHTSA
PREVIOUS [2012-02-08] NTSB · Recommendation Reiteration | duplicate: False
    From the greensheet issuing safety recommendations H-11-36 through H-11-38, which were issued as a result of the August 5, 2010 highway accident in Gray Summit, Missouri: The NTSB concluded that FCW s
THIS     [2012-02-08] NTSB · Report Reclassification
   … -fatality accident, when a truck-tractor semitrailer combination unit rear-ended and overrode several passenger vehicles on I-44 near Miami, Oklahoma, the NTSB reiterated Safety Recommendations H-01-6 and -7 to NHTSA and reclassified their status to “Open—Unacceptable Response.
A-96-110 → FAA
PREVIOUS [1997-04-23] NTSB · NPRM Response | duplicate: False
 The National Transportation Safety Board has reviewed your notice of proposed rulemaking, "Airworthiness Directives; Boei

**What the 10 showed:** 7,093 graded NTSB letters come right after another NTSB letter, and 2,036 of those
are exact duplicates. The rest weren't reply decisions at all:
- **duplicates**, exact or near (same day, one copy cut short)
- grades from **accident reports** (reiterations, reclassifications) and **superseded** recommendations
- grades based on a **proposed rule** (NPRM) the agency published, not a letter
- **"status stays the same, please send an update"** nudges after long silence
- grades for **other** recommendations in a letter copied into this thread (our ID rules already drop those)

None of them was "NTSB acknowledged the reply first and graded it later", so there's nothing to recover here.

## Findings and decisions

| # | What we found | How many | Decision |
|---|---|---|---|
| 1 | Same letter stored twice in a thread (exact or near-duplicate) | 6,511+ | Remove duplicates, keep the longer copy |
| 2 | NTSB's original letter is missing, so the first letter is the org's reply | 10,186 of 18,904 threads; 4,887 start 2+ years after the ID year | Day 0 = Jan 1 of the year in the ID (approx), plus "days since previous letter" |
| 3 | Old letters are ALL CAPS | 56% of 1980s, 80% of 1990s, ~0% after 2010 | Convert ALL CAPS letters to normal sentence case |
| 4 | Dates in the future (typos like year 2500) | 2 | Skip letters dated after today |
| 5 | NTSB's answer uses info that isn't in the reply (meetings, calls, attachments, documents) | ~1,029 + 577 of 12,738 (rough check) | Keep for grading, tag it, leave it out of letter-writing training |
| 6 | NTSB grades with no new reply (reports, rules, nudges, supersedes) | ~7,000 | Not practice questions; they stay in the history as context |
| 7 | Open, no reply, 90+ days since the last letter | 755 | Demo cases for overdue alerts + reminder letters |
| 8 | Mentioned again after a new crash while the recipient is silent (e.g. H-25-025) | — | Demo cases for escalation |
| — | Undated letters | 1,869 | Skipped (already) |
| — | Batch letters (3+ recommendations) | 12% of NTSB letters | Handled by the ID rules (already) |
| — | Sent to several recipients | ~600 recommendations, up to 78 | Peer comparison works |
| — | Threads with 20+ letters / letters over 20k characters | 288 / 260 | Paging in `read_letter` handles them |